In [ ]:
from typing import Any
from matplotlib import pyplot as plt
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import resnet18
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import ToTensor
from tqdm.auto import tqdm
import misc

Modify dataset and model for pretraining.

In [ ]:
class CifarForScan(CIFAR10):
    def __getitem__(self, idx: int):
        img, _ = super().__getitem__(idx)
        return idx, img

In [ ]:
class ResnetForScan(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self._resnet = resnet18(num_classes=10)
        self._extractor = create_feature_extractor(
            self._resnet,
            return_nodes=['avgpool']
        )
    
    def forward(self, img: torch.Tensor) -> torch.Tensor:
        return self._extractor(img)['avgpool'].view(-1, 512)

Forward pass

In [ ]:
train_data = CifarForScan(
    root='C:/Users/chiwe/Data',
    train=True,
    transform=ToTensor()
)
valid_data = CifarForScan(
    root='C:/Users/chiwe/Data',
    train=False,
    transform=ToTensor()
)
train_loader = DataLoader(train_data, batch_size=100, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=100, shuffle=False)
model = ResnetForScan().cuda()
optimizer = optim.SGD(model.parameters(), lr=1e-1, momentum=0.9, weight_decay=1e-4)

# instance discrimination memory bank
torch.manual_seed(0)
memory = torch.randn(len(train_data), 512, device='cuda')
memory = nn.functional.normalize(memory, p=2, dim=-1)

loss_history = []
progbar, num_batches = tqdm(range(5)), len(train_loader)
for _ in progbar:
    for i_batch, (idcs, imgs) in enumerate(train_loader):
        # forward pass
        features = model(imgs.cuda())
        features = nn.functional.normalize(features, p=2, dim=-1)
        from_mem = memory[idcs]
        logits = features @ from_mem.T
        loss = logits.softmax(dim=-1).diag().log().mean().neg()

        if loss.isnan():
            raise ValueError('nan loss')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_history.append(loss.detach().item())

        # update memory bank
        memory[idcs] = features.detach()

        # logging
        progbar.set_postfix({'batch': f'{i_batch}/{num_batches}'})

# plot training loss
fig, axes = plt.subplots()
axes.plot(loss_history)
axes.grid()
fig.tight_layout()

t-SNE colored by class label after pretraining

In [ ]:
model.eval()

valid_data = CIFAR10(
    root='C:/Users/chiwe/Data',
    train=False,
    transform=ToTensor()
)
valid_loader = DataLoader(valid_data, batch_size=100, shuffle=False)


with torch.no_grad():
    # train_features = []
    # for _, imgs in tqdm(train_loader):
    #     train_features.append(model(imgs.cuda()).cpu())
    
    valid_features = []
    valid_labels = []
    for imgs, labels in tqdm(valid_loader):
        valid_features.append(model(imgs.cuda()).cpu())
        valid_labels.append(labels)

valid_features = torch.cat(valid_features)
valid_labels = torch.cat(valid_labels)

In [ ]:
fig, axes = misc.gen_tsne_plot(
    valid_features,
    valid_labels
)